# Two-Point Effective Mass Example

Standalone effective-mass workflow for the canonical `l64c64a076_m140` example.
It reads correlator CSV files, applies folding, bootstraps the data,
writes effective-mass tables, and saves one plot per momentum.


## Imports / Setup


In [1]:
from pathlib import Path
import sys

REPO_ROOT = None
for parent in Path.cwd().resolve().parents:
    if parent.name == "lat-hadron-analysis-DA":
        REPO_ROOT = parent
        break
if REPO_ROOT is None:
    raise RuntimeError("could not locate repository root")
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np

import matplotlib
matplotlib.rcParams.update({
    "font.family": "Times New Roman",
    "mathtext.fontset": "custom",
    "mathtext.rm": "Times New Roman",
    "mathtext.it": "Times New Roman:italic",
    "mathtext.bf": "Times New Roman:bold",
})

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_effective_mass_input_text,
    run_effective_mass_from_notebook,
    validate_effective_mass_notebook_config,
)
from lqcd_analysis.two_point.plotting import plot_effective_mass


## User Inputs


In [2]:
EXAMPLE_DATA = REPO_ROOT / 'examples' / 'l64c64a076_m140' / 'data' / 'c2pt_csv'
EXAMPLE_OUTPUTS = REPO_ROOT / 'examples' / 'l64c64a076_m140' / 'analysis' / '0-effective-mass' / 'results_effective_mass_notebook'

workflow_config = {
    'title_pattern': 'l64c64a076_m140_fit_k6_pz*',
    "ns": 64,
    "nt": 64,
    "lattice_spacing_fm": 0.076,
    'c2pt': str(EXAMPLE_DATA / 'c2pt_5_5_k6_pz*_real.csv'),
    'pzlist': [5, 6, 7, 8, 9, 10],
    'fold_t': 'periodic',
    'tsrange': [0, 13],
    'model': 'symmetric',
    'binsize': 10,
    'bootstrap_samples': 200,
    'bootstrap_size': 200,
    'seed': 2026,
    'results_dir': str(EXAMPLE_OUTPUTS),
}


- `title_pattern`: output title pattern.
- `c2pt`: correlator CSV path or wildcard pattern.
- `pzlist`: momenta to analyze.
- `fold_t`: folding mode before bootstrapping.
- `tsrange`: retained time range.
- `model`: `symmetric` here because the example data are periodic.
- `binsize`, `bootstrap_samples`, `bootstrap_size`, `seed`: bootstrap controls.
- `results_dir`: output directory used by both notebook and input-file runs.
- The workflow writes effective-mass tables and then the notebook saves PDF plots from those tables.


In [3]:
validated = validate_effective_mass_notebook_config(workflow_config)
print(pretty_print_config(validated))
print(render_effective_mass_input_text(workflow_config))


"EffectiveMassInput(title_pattern='l64c64a076_m140_fit_k6_pz*', ns=64, nt=64, lattice_spacing_fm=0.076, correlator_path_pattern='/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/c2pt_csv/c2pt_5_5_k6_pz*_real.csv', pzlist=(5, 6, 7, 8, 9, 10), fold_t='periodic', tsrange=(0, 13), model='symmetric', binsize=10, bootstrap_samples=200, bootstrap_size=200, seed=2026, results_dir=PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/0-effective-mass/results_effective_mass_notebook'))"
l64c64a076_m140_fit_k6_pz* 64 64 0.076
c2pt /Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/c2pt_csv/c2pt_5_5_k6_pz*_real.csv
pzlist 5 6 7 8 9 10
fold_t periodic
model symmetric
tsrange 0 13
binsize 10
bootstrap_samples 200
bootstrap_size 200
seed 2026
results_dir /Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/0-effective-mass/results_effective_mass_notebook



## Run Workflow


In [4]:
outputs = run_effective_mass_from_notebook(workflow_config)
outputs


[PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/0-effective-mass/results_effective_mass_notebook/l64c64a076_m140_fit_k6_pz5/tables/l64c64a076_m140_fit_k6_pz5_symmetric_effective_mass.txt'),
 PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/0-effective-mass/results_effective_mass_notebook/l64c64a076_m140_fit_k6_pz6/tables/l64c64a076_m140_fit_k6_pz6_symmetric_effective_mass.txt'),
 PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/0-effective-mass/results_effective_mass_notebook/l64c64a076_m140_fit_k6_pz7/tables/l64c64a076_m140_fit_k6_pz7_symmetric_effective_mass.txt'),
 PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/0-effective-mass/results_effective_mass_notebook/l64c64a076_m140_fit_k6_pz8/tables/l64c64a076_m140_fit_k6_pz8_symmetric_effective_mass.txt'),
 PosixPath('/Users/xiang/Desktop/codes/lat-hadro

## Plot Results

Use the tables produced above to save one effective-mass plot per momentum.


In [5]:
plot_paths = []
for table_path in outputs:
    table = np.loadtxt(table_path)
    if table.ndim == 1:
        table = table[None, :]
    times = table[:, 0]
    meff_mean = table[:, 1]
    meff_err = table[:, 2]
    dataset_dir = table_path.parent.parent
    plots_dir = dataset_dir / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)
    plot_path = plots_dir / f"{table_path.stem}.pdf"
    plot_effective_mass(plot_path, times, meff_mean, meff_err, title=table_path.stem)
    plot_paths.append(plot_path)
plot_paths


[PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/0-effective-mass/results_effective_mass_notebook/l64c64a076_m140_fit_k6_pz5/plots/l64c64a076_m140_fit_k6_pz5_symmetric_effective_mass.pdf'),
 PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/0-effective-mass/results_effective_mass_notebook/l64c64a076_m140_fit_k6_pz6/plots/l64c64a076_m140_fit_k6_pz6_symmetric_effective_mass.pdf'),
 PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/0-effective-mass/results_effective_mass_notebook/l64c64a076_m140_fit_k6_pz7/plots/l64c64a076_m140_fit_k6_pz7_symmetric_effective_mass.pdf'),
 PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/0-effective-mass/results_effective_mass_notebook/l64c64a076_m140_fit_k6_pz8/plots/l64c64a076_m140_fit_k6_pz8_symmetric_effective_mass.pdf'),
 PosixPath('/Users/xiang/Desktop/codes/lat-hadron-an